In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "credit_risk"

target_table = f"{CATALOG}.{SCHEMA}.gold_credit_modeling_base_target"
features_table = f"{CATALOG}.{SCHEMA}.gold_credit_modeling_base_features"
abt_table = f"{CATALOG}.{SCHEMA}.abt_credit_default"

target_df = (
    spark.table(target_table)
    .withColumn("date_position", F.col("date_position").cast("date"))
    .withColumn("flag_default", F.col("flag_default").cast("int"))
)

features_df = (
    spark.table(features_table)
    .withColumn("date_position", F.col("date_position").cast("date"))
)

feature_columns = [
    column
    for column in features_df.columns
    if column not in {"Customer_ID", "date_position"}
]

abt_df = (
    target_df
    .select("Customer_ID", "date_position", "flag_default")
    .join(
        features_df.select("Customer_ID", "date_position", *feature_columns),
        on=["Customer_ID", "date_position"],
        how="inner",
    )
)

(
    abt_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(abt_table)
)

In [0]:
from pyspark.sql import functions as F

abt_df.agg(
    F.min("date_position").alias("first_reference_date"),
    F.max("date_position").alias("last_reference_date"),
    F.countDistinct("date_position").alias("reference_dates"),
    F.count("*").alias("total_rows"),
).display()

(
    abt_df
    .groupBy("date_position")
    .agg(
        F.count("*").alias("total_customers"),
        F.round(F.avg("flag_default") * 100, 2).alias("default_rate_pct"),
    )
    .orderBy("date_position")
    .display()
)